## Tracing Requests

# Steps to activate tracing 

"wget https://raw.githubusercontent.com/triton-inference-server/server/main/docs/examples/python/trace_summary.py"

"nano summer2025/docker/docker-compose.yaml"

"command: tritonserver --trace-file=/tmp/trace.json --trace-rate=1 --trace-count=20000 --trace-log-frequency=4000 --trace-level=TIMESTAMPS --model-repository=/models"
 * trace-rate = 1 -> so that we trace every request
 * trace-count => the server will stop tracing after that many requests
 * trace-log-frequency => should match how many requests we send in the benchmark (request-rate*request-count) -> each trace file will contain this many request traces

Then build and force recreate the server:

"docker compose -f ~/summer2025/docker/docker-compose.yaml build triton_server"

"docker compose -f ~/summer2025/docker/docker-compose.yaml up triton_server --force-recreate -d"

To see the trace files logged:

"docker exec triton_server ls -alstr /tmp/"

To copy them to the jupyter interface: 

"docker cp triton_server:/tmp/trace.json.1 ."

"docker cp trace.json.1 jupyter:/home/jovyan/work/trace_r6_1.json"

In [ ]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

In [ ]:
# Warmup requests
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f warmup.csv

In [ ]:
# Warmup stats
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

In [ ]:
# Trace 1 requests
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f warmup.csv

In [ ]:
# Warmup + Trace 1 stats
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats